In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets sentencepiece nltk langdetect pandas scikit-learn
!pip install clean-text


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 53.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=6549002ff657c18100b3f72e9729675ba413366aa9e315d5199922c9f6bdcfe5
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.4/175.4 kB 14.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
  Created wheel for emoji: filename=emoji-1.7.0-py3-none-any.whl size=171031 sha256=3fefd6d841c4cc314ad88ba590ddd88cfeb3ec1726a5bbdb7b42da02d3b6e07d
  Stored in directory: /root/.cache/pip/wheels/e0/8c/e0/294d2e4ea0e55792bfc99b6b263e4a0511443da7b69af67688
Successfully built emoji


In [ ]:
import re
import unicodedata
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.utils.class_weight import compute_class_weight
import torch
import os
from cleantext import clean
from langdetect import detect
import nltk
nltk.download('wordnet')


[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
# Path to your Excel file
input_excel = '/content/drive/MyDrive/CLEANED_output_combined.csv.xlsx'

# Read the Excel file into a DataFrame using openpyxl engine
df = pd.read_excel(input_excel, engine='openpyxl')

# Display the first few rows to confirm successful loading
print(df.head())

# Check for expected columns
if 'department' not in df.columns or 'text' not in df.columns:
    raise ValueError("Expected columns 'department' and 'text' not found in input Excel file.")


  department                                               text
0    Finance  bid details/ bid end date/time/ / 20-02-2025 1...
1    Finance  , no startup exemption for years of experience...
2    Finance  for technical clarifications during technical ...
3    Finance  financial document indicating price breakup re...
4    Finance  and its subsequent orders/notifications issued...


In [ ]:
# Normalize department names: strip whitespace and convert to lowercase
df['department_normalized'] = df['department'].astype(str).str.strip().str.lower()

# Create label map based on normalized names
unique_departments = sorted(df['department_normalized'].unique())
label_map = {dept: idx for idx, dept in enumerate(unique_departments)}

# Add normalized label column
df['label'] = df['department_normalized'].map(label_map)

# For displaying original department to label mapping, choose representative original values:
# You can map normalized to first encountered original department string per normalized key
repr_original = df.groupby('department_normalized')['department'].first().to_dict()

print("Normalized Department to Label Map:")
for norm_dept, idx in label_map.items():
    print(f"{repr_original[norm_dept]} ({norm_dept}) : {idx}")


Normalized Department to Label Map:
Engineering  (engineering) : 0
Finance (finance) : 1
HR (hr) : 2
Maintenance  (maintenance) : 3
Operations (operations) : 4


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(),
    test_size=0.2, random_state=42, stratify=df['label'].tolist()
)


In [ ]:
checkpoint = 'ai4bharat/indic-bert'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_batch(texts, tokenizer, max_length=256):
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors='pt'
    )

X_train = tokenize_batch(train_texts, tokenizer)
X_val = tokenize_batch(val_texts, tokenizer)
y_train = torch.tensor(train_labels)
y_val = torch.tensor(val_labels)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

In [ ]:
from torch.utils.data import Dataset

class SimpleDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k:v[idx] for k,v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SimpleDataset(X_train, y_train)
val_dataset = SimpleDataset(X_val, y_val)


In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels.numpy() if isinstance(train_labels, torch.Tensor) else train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class Weights:", class_weights)


Class Weights: tensor([0.4402, 0.7692, 1.3761, 2.2237, 3.9673])


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_map),
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
)

# Freeze first 1-2 layers (optional)
for name, param in model.named_parameters():
    if 'encoder.layer.0' in name or 'encoder.layer.1' in name:
        param.requires_grad = False


pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/indic-bert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import Trainer, TrainingArguments

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss



In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}


In [ ]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/indicbert_multilingual_model',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.1,
    warmup_steps=500,
    logging_steps=50,
    eval_steps=50,
    save_steps=500,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    gradient_accumulation_steps=2,
    dataloader_drop_last=True,
    fp16=True,
    seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-2584342335.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kushagrasingh3062 (kushagrasingh3062-iiit-nagpur-official) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/tmp/ipython-input-952648790.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels[idx])


Step,Training Loss,Validation Loss,Accuracy
50,1.611500,1.611063,0.259983
100,1.610900,1.610953,0.259983
150,1.610900,1.610670,0.259983
200,1.611200,1.609992,0.259983
250,1.609100,1.608919,0.259983
300,1.608300,1.608333,0.259115
350,1.608300,1.607743,0.259838
400,1.606300,1.606117,0.259983
450,1.602600,1.599592,0.516493
500,1.572300,1.565637,0.507812


/tmp/ipython-input-952648790.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels[idx])
/tmp/ipython-input-952648790.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels[idx])
/tmp/ipython-input-952648790.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels[idx])
/tmp/ipython-input-952648790.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.d

TrainOutput(global_step=3456, training_loss=1.1365459148806554, metrics={'train_runtime': 2734.489, 'train_samples_per_second': 40.449, 'train_steps_per_second': 1.264, 'total_flos': 1321861323423744.0, 'train_loss': 1.1365459148806554, 'epoch': 4.0})

In [ ]:
from sklearn.metrics import accuracy_score

eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

train_preds_output = trainer.predict(train_dataset)
train_preds = train_preds_output.predictions.argmax(axis=-1)
train_labels_arr = train_preds_output.label_ids
train_acc = accuracy_score(train_labels_arr, train_preds)
print(f"Training accuracy: {train_acc:.4f}")

# For test dataset, if available
# test_preds_output = trainer.predict(test_dataset)
# test_preds = test_preds_output.predictions.argmax(axis=-1)
# test_labels_arr = test_preds_output.label_ids
# test_acc = accuracy_score(test_labels_arr, test_preds)
# print(f"Test accuracy: {test_acc:.4f}")


/tmp/ipython-input-952648790.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels[idx])


Evaluation results: {'eval_loss': 0.9707226157188416, 'eval_accuracy': 0.7501446759259259, 'eval_runtime': 25.2231, 'eval_samples_per_second': 274.075, 'eval_steps_per_second': 17.167, 'epoch': 4.0}


/tmp/ipython-input-952648790.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels[idx])


Training accuracy: 0.7608


In [ ]:
save_dir = '/content/drive/MyDrive/indicbert_multilingual_model/final'
os.makedirs(save_dir, exist_ok=True)
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved final model and tokenizer to:", save_dir)

Saved final model and tokenizer to: /content/drive/MyDrive/indicbert_multilingual_model/final
